### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

from llama_cloud_services import LlamaParse
from llama_index.core import Document
import re

from dotenv import load_dotenv
load_dotenv()

llama_api_key = os.getenv("LLAMA_CLOUD_API_KEY")

c:\Users\kirth\OneDrive\Desktop\AI_Software_Engineer_Kirthika\Github_Projects_for_AI\Agentic_AI_Overall\RAG_For_RACV_PolicyDocs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\kirth\AppData\Local\Temp\ipykernel_41620\3916170459.py:5: DeprecationWarning: This package (llama-cloud-services) is deprecated and will be maintained until May 1, 2026. Please migrate to the new package: pip install llama-cloud>=1.0 (https://github.com/run-llama/llama-cloud-py). The new package provides the same functionality with improved performance and support.
  from llama_cloud_services import LlamaParse


In [2]:
import asyncio
import nest_asyncio
from llama_parse import LlamaParse
from llama_index.core import Document

# Patch the loop environment
nest_asyncio.apply()

async def process_all_pdfs_async():
    parsing_instructions = """
    This document is an insurance policy or roadside assistance terms and conditions document. 
    It contains benefit tables, eligibility conditions, coverage limits, and exclusion clauses. 
    Extract all tables as markdown tables preserving every row and column. 
    Preserve the relationship between coverage items and their corresponding limits. 
    Capture all specific quantities, distances, dollar amounts, and time limits exactly as written. 
    Do not merge separate clauses into single paragraphs.
    DO NOT ADD ANYTHING TO THE PDF DATA.
    """
    
    # Re-initialize the parser locally inside the running event loop context
    parser = LlamaParse(
        result_type="markdown",
        system_prompt=parsing_instructions,
        verbose=True
    )
    
    pdf_files = [
        "../data/pdfs/era-terms-and-conditions-2023-24.pdf",
        "../data/pdfs/motor-insurance-complete-care-pds-current.pdf"
    ]
    
    all_documents = []
    
    # Explicitly await files one by one to keep the async buffer clean
    for pdf_path in pdf_files:
        try:
            print(f"\nParsing via Cloud API: {pdf_path}")
            
            # Using aload_data keeps the background network loop active natively
            parsed_sections = await parser.aload_data(pdf_path)
            
            section_count = 0
            for doc in parsed_sections:
                clean_text = doc.text.replace('\t', ' ')
                clean_text = re.sub(r' +', ' ', clean_text)
                
                # Skip near-empty sections
                if len(clean_text.strip()) < 50:
                    continue
                
                all_documents.append(Document(
                    text=clean_text,
                    metadata={
                        "source_file": os.path.basename(pdf_path),
                        "file_type": "pdf"
                    }
                ))
                section_count += 1
                
            print(f" ✓ {os.path.basename(pdf_path)} — Loaded {section_count} cleaned chunks.")
            
        except Exception as e:
            print(f" ✗ Error parsing {pdf_path}: {e}")
            
    print(f"\nTotal clean chunks loaded globally: {len(all_documents)}")
    
    if all_documents:
        print("\nFirst true document preview:")
        print(all_documents[0].text[:500])
        
    return all_documents

# Trigger execution using the active asyncio loop framework
all_pdf_documents = asyncio.run(process_all_pdfs_async())


C:\Users\kirth\AppData\Local\Temp\ipykernel_41620\795833671.py:3: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse



Parsing via Cloud API: ../data/pdfs/era-terms-and-conditions-2023-24.pdf
Started parsing the file under job_id 1edc77a8-cb75-4661-8aae-3d9ac89aaab2
 ✓ era-terms-and-conditions-2023-24.pdf — Loaded 15 cleaned chunks.

Parsing via Cloud API: ../data/pdfs/motor-insurance-complete-care-pds-current.pdf
Started parsing the file under job_id 253b4770-c8b6-422f-a682-cf895fa3b73e
 ✓ motor-insurance-complete-care-pds-current.pdf — Loaded 52 cleaned chunks.

Total clean chunks loaded globally: 67

First true document preview:

```markdown
| Section | Page |
|-------------------------------------------------------------|------|
| Introduction | 4 |
| RACV Emergency Roadside Assistance | 6 |
| Obtaining Assistance | 6 |
| Proof of ERA product holding | 6 |
| ERA Products Features Table | 7 |
| 1. RACV Total Care | 8 |
| 1.1 Emergency Roadside Assistance | 8 |
| 1.2 Extended Benefits | 8 |
| General Benefits | 8 |
| Benefits when under 100km from home address | 9 |
| 1.3 Benefits when over 100km fr

In [3]:
all_pdf_documents[:5]

[Document(id_='8a603ccb-bb3b-49c4-8d9e-824dff899663', embedding=None, metadata={'source_file': 'era-terms-and-conditions-2023-24.pdf', 'file_type': 'pdf'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='\n```markdown\n| Section | Page |\n|-------------------------------------------------------------|------|\n| Introduction | 4 |\n| RACV Emergency Roadside Assistance | 6 |\n| Obtaining Assistance | 6 |\n| Proof of ERA product holding | 6 |\n| ERA Products Features Table | 7 |\n| 1. RACV Total Care | 8 |\n| 1.1 Emergency Roadside Assistance | 8 |\n| 1.2 Extended Benefits | 8 |\n| General Benefits | 8 |\n| Benefits when under 100km from home address | 9 |\n| 1.3 Benefits when over 100km from home address | 9 |\n| Personal Benefits | 10 |\n| 2. RACV Extra Care | 10 |\n| 2.1 Emergency Roadside Assistance | 10 |\n| 2.2 Extended Benefits

In [4]:
from llama_index.core.node_parser import MarkdownNodeParser

def split_documents(documents):
    """
    Splits LlamaParse markdown documents into optimal RAG chunks (Nodes)
    while preserving markdown tables and structural headers intact.
    """
    # 1. Initialize the Markdown-aware parser
    # It automatically groups content logically by headers and markdown blocks
    markdown_parser = MarkdownNodeParser()
    
    print(f"Splitting {len(documents)} parsed document sections...")
    
    # 2. Get chunks (LlamaIndex calls these 'Nodes')
    split_nodes = markdown_parser.get_nodes_from_documents(documents)
    
    print(f"Successfully split into {len(split_nodes)} optimized RAG chunks.")
    
    # 3. Show a safe example chunk
    if split_nodes:
        print(f"\nExample chunk:")
        print(f"Content:\n{split_nodes[0].text[:300]}...")
        print(f"Metadata: {split_nodes[0].metadata}")
        
    return split_nodes

# Execute the chunking step
llama_chunks = split_documents(all_pdf_documents)


Splitting 67 parsed document sections...
Successfully split into 67 optimized RAG chunks.

Example chunk:
Content:
```markdown
| Section | Page |
|-------------------------------------------------------------|------|
| Introduction | 4 |
| RACV Emergency Roadside Assistance | 6 |
| Obtaining Assistance | 6 |
| Proof of ERA product holding | 6 |
| ERA Products Features Table | 7 |
| 1. RACV Total Care | 8 |
| 1.1...
Metadata: {'source_file': 'era-terms-and-conditions-2023-24.pdf', 'file_type': 'pdf', 'header_path': '/'}


In [5]:
type(llama_chunks)

list

### Embedding and VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "nomic-ai/nomic-embed-text-v1.5"):
        """Initialize embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings"""

        self.model_name =  model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model =  SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimesions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """ Generate embeddings for a list of texts

            Args:
                texts: List of text strings to embed
            return:
                numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## Initilize the Embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: nomic-ai/nomic-embed-text-v1.5


Loading weights: 100%|██████████| 112/112 [00:00<00:00, 1585.00it/s]


Model loaded successfully. Embedding dimesions: 768


C:\Users\kirth\AppData\Local\Temp\ipykernel_41620\3875187586.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimesions: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
            
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        Args:
            documents: List of LlamaIndex TextNode / Document objects
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
            
        print(f"Adding {len(documents)} documents to vector store...")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # 1. Generate clean unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # 2. Extract text (Handles both LangChain page_content and LlamaIndex text properties)
            text_content = getattr(doc, "text", getattr(doc, "page_content", ""))
            documents_text.append(text_content)
            
            # 3. Clean and sanitize metadata for ChromaDB compatibility
            raw_metadata = getattr(doc, "metadata", {}) or {}
            clean_metadata = {}
            for k, v in raw_metadata.items():
                if isinstance(v, (str, int, float, bool)):
                    clean_metadata[k] = v
                else:
                    clean_metadata[k] = str(v) # Convert complex types (lists/dicts) to string
                    
            clean_metadata['doc_index'] = i
            clean_metadata['content_length'] = len(text_content)
            metadatas.append(clean_metadata)
            
            # 4. Append vector embedding list
            embeddings_list.append(embedding.tolist())
            
        # Add everything to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

# Initialize the corrected vector store
vectorstore = VectorStore()


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 67


In [9]:
llama_chunks

[TextNode(id_='f1cd3195-cacc-41af-9405-04cedd8a3a4e', embedding=None, metadata={'source_file': 'era-terms-and-conditions-2023-24.pdf', 'file_type': 'pdf', 'header_path': '/'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='8a603ccb-bb3b-49c4-8d9e-824dff899663', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'source_file': 'era-terms-and-conditions-2023-24.pdf', 'file_type': 'pdf'}, hash='5b7ae89570de9cbd206e32eebca5e07d92b3022419cd8debf8306fa4e63f6724')}, metadata_template='{key}: {value}', metadata_separator='\n', text='```markdown\n| Section | Page |\n|-------------------------------------------------------------|------|\n| Introduction | 4 |\n| RACV Emergency Roadside Assistance | 6 |\n| Obtaining Assistance | 6 |\n| Proof of ERA product holding | 6 |\n| ERA Products Features Table | 7 |\n| 1. RACV Total Care | 8 |\n| 1.1 Emergency Roadside Assistance | 8 |\n| 1.2 Extended Benefits | 8 |\n| Gen

In [10]:
texts=[doc.text for doc in llama_chunks]
texts

['```markdown\n| Section | Page |\n|-------------------------------------------------------------|------|\n| Introduction | 4 |\n| RACV Emergency Roadside Assistance | 6 |\n| Obtaining Assistance | 6 |\n| Proof of ERA product holding | 6 |\n| ERA Products Features Table | 7 |\n| 1. RACV Total Care | 8 |\n| 1.1 Emergency Roadside Assistance | 8 |\n| 1.2 Extended Benefits | 8 |\n| General Benefits | 8 |\n| Benefits when under 100km from home address | 9 |\n| 1.3 Benefits when over 100km from home address | 9 |\n| Personal Benefits | 10 |\n| 2. RACV Extra Care | 10 |\n| 2.1 Emergency Roadside Assistance | 10 |\n| 2.2 Extended Benefits | 11 |\n| General Benefits | 11 |\n| Benefits when under 100km from home address | 11 |\n| Benefits when over 100km from home address | 11 |\n| 3. RACV Roadside Care | 12 |\n| 3.1 Emergency Roadside Assistance | 12 |\n| 3.2 Extended Benefits | 13 |\n| General Benefits | 13 |\n| Benefits when under 100km from home address | 13 |\n| Benefits when over 100km fr

In [11]:
### Convert the text to embeddings
texts=[doc.text for doc in llama_chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(llama_chunks,embeddings)

Generating embeddings for 67 texts...


Batches: 100%|██████████| 3/3 [02:47<00:00, 55.79s/it] 


Generated embeddings with shape: (67, 768)
Adding 67 documents to vector store...
Successfully added 67 documents to vector store
Total documents in collection: 134


### Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int=5, score_threshold: float= 0.0) -> List[Dict[str, Any]]:
        """
            Retrieve relevant documents for a query

            Args:
                query: The search query
                top_k: Number of top results to return
                score_threshold: Minimum similarity score threshold
            Returns:
                List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for the query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embeddings.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results ['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i+1
                        })
                    
                print(f"Retrieved {len(retrieved_docs)} docs (after filtering)")
                return retrieved_docs
                
            else:
                print(f"Error during retrieval: {e}")
                return []
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)                   

In [13]:
rag_retriever

In [14]:
rag_retriever.retrieve("query= what is the SPDS Edition 1?")

Retrieving documents for the query: query= what is the SPDS Edition 1?
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]

Generated embeddings with shape: (1, 768)
Retrieved 0 docs (after filtering)


[]

In [15]:
rag_retriever.retrieve("Does RACV cover for accidental damage to your vehicle in Complete care?")

Retrieving documents for the query: Does RACV cover for accidental damage to your vehicle in Complete care?
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.31it/s]

Generated embeddings with shape: (1, 768)
Retrieved 0 docs (after filtering)


[]

In [16]:
rag_retriever.retrieve("How many litres of petrol does RACV provide in an emergency?")


Retrieving documents for the query: How many litres of petrol does RACV provide in an emergency?
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.89it/s]

Generated embeddings with shape: (1, 768)
Retrieved 0 docs (after filtering)


[]

In [17]:
rag_retriever.retrieve("Tell me about Claiming under my Policy ?")


Retrieving documents for the query: Tell me about Claiming under my Policy ?
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.02it/s]

Generated embeddings with shape: (1, 768)
Retrieved 0 docs (after filtering)


[]

### Integration Vectordb Context pipeline With LLM output

In [18]:
### Simple RAG pipeline with Groq LLM
from langchain_anthropic import ChatAnthropic
import os
from dotenv import load_dotenv
load_dotenv()

claude_api_key = os.getenv("CLAUDE_API_KEY")

llm = ChatAnthropic(anthropic_api_key=claude_api_key, model="claude-sonnet-4-5-20250929", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)

    context="\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt])
    return response.content

In [19]:
answer = rag_simple("How many litres of petrol does RACV provide in an emergency?", rag_retriever, llm)
print(answer)

Retrieving documents for the query: How many litres of petrol does RACV provide in an emergency?
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.76it/s]

Generated embeddings with shape: (1, 768)
Retrieved 0 docs (after filtering)
No relevant context found to answer the question.


### Enhanced RAG Pipeline Features

In [20]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("How many litres of petrol does RACV provide in an emergency?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for the query: How many litres of petrol does RACV provide in an emergency?
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.42it/s]

Generated embeddings with shape: (1, 768)
Retrieved 0 docs (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [21]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for the query: what is attention is all you need
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.30it/s]


Generated embeddings with shape: (1, 768)
Retrieved 0 docs (after filtering)

Final Answer: No relevant context found.
Summary: I cannot provide a summary because there is no actual answer or content to summarize. The statement "No relevant context found" simply indicates that no information was available to address whatever question was asked.
History: {'question': 'what is attention is all you need', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'I cannot provide a summary because there is no actual answer or content to summarize. The statement "No relevant context found" simply indicates that no information was available to address whatever question was asked.'}
